# Colab End-to-End Distillation Testing: Stage 0 + Stage 1

**Objective**: Validate the complete Hybrid Mamba-xLSTM distillation pipeline before A100 production.

- **Stage 0**: LM pretraining with BioMedLM knowledge distillation (5000 steps)
- **Stage 1**: SimCSE contrastive learning with PubMedBERT KD (2000 steps)
- **Total time**: ~2-3 hours on T4 (including setup, pytest, and validation)

Uses production configs: `stage0_biomedlm.yaml` + `stage1_pubmedbert.yaml`

## Phase 0: Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted.")

In [ ]:
import os

os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_DATASETS_CACHE'] = '/content/hf_cache/datasets'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb=128'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

print("Environment variables set:")
print(f"  HF_HOME: {os.environ['HF_HOME']}")
print(f"  PYTORCH_CUDA_ALLOC_CONF: {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")

In [ ]:
import subprocess
import os

repo_path = '/content/hybrid_model_mamba_xlstm'
branch = 'a100_70m_baseline'

if not os.path.exists(repo_path):
    cmd = f'git clone --branch {branch} https://github.com/krishankb-de/hybrid_model_mamba_xlstm.git {repo_path}'
    print(f"Cloning repository...")
    subprocess.run(cmd, shell=True, check=True)
    print("Repository cloned successfully.")
else:
    print(f"Repository exists. Updating...")
    os.chdir(repo_path)
    subprocess.run(f'git fetch origin {branch}', shell=True)
    subprocess.run(f'git checkout {branch}', shell=True)
    subprocess.run('git pull origin', shell=True)
    print(f"Updated to latest {branch}")

os.chdir(repo_path)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import subprocess

print("Installing dependencies...")
subprocess.run('pip install -q -e .', shell=True, check=True)
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
print("Dependencies installed successfully.")

In [ ]:
import torch

print("PyTorch Information:")
print(f"  PyTorch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  Device name: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  Total VRAM: {total_mem:.1f} GB")
else:
    print("  WARNING: CUDA not available!")

## Phase 1: Verify Test Infrastructure

In [ ]:
import subprocess

print("Running pytest on test_encoder_pooling.py...")
result = subprocess.run('pytest tests/test_encoder_pooling.py -x -v', shell=True)
if result.returncode == 0:
    print("\n✓ test_encoder_pooling.py PASSED")
else:
    raise RuntimeError("Test failed.")

In [ ]:
import subprocess

print("Running smoke_test_distill.py...")
result = subprocess.run('python scripts/smoke_test_distill.py', shell=True)
if result.returncode == 0:
    print("\n✓ smoke_test_distill.py PASSED")
else:
    raise RuntimeError("Smoke test failed.")

## Phase 2: Pre-download Teachers & Data

In [ ]:
from transformers import AutoModelForCausalLM
import torch

print("Pre-caching BioMedLM (2.7B)...")
biomedlm = AutoModelForCausalLM.from_pretrained(
    'stanford-crfm/BioMedLM',
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
print("✓ BioMedLM cached successfully.")
del biomedlm
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

print("Pre-caching PubMedBERT (110M)...")
pubmedbert_name = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'
model = AutoModel.from_pretrained(pubmedbert_name, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
tokenizer = AutoTokenizer.from_pretrained(pubmedbert_name)
print("✓ PubMedBERT cached successfully.")
del model, tokenizer
torch.cuda.empty_cache()

## Phase 3: Stage 0 LM Pretraining (5k steps, ~80-100 min)

In [ ]:
import subprocess

stage0_cmd = '''
python scripts/train_stage0_distill.py \\
    --config-name config_70m \\
    dataset=pubmed \\
    trainer=colab_single_gpu \\
    distill=stage0_biomedlm \\
    trainer.max_steps=5000 \\
    trainer.accumulate_grad_batches=4 \\
    dataset.batch_size=8 \\
    trainer.log_every_n_steps=50 \\
    trainer.precision=16-mixed \\
    experiment_name=colab_stage0_kd_biomedlm_test \\
    output_dir=./outputs/colab_stage0_kd_biomedlm_test \\
    wandb.enabled=false
'''

print("="*80)
print("STAGE 0: LM Pretraining (BioMedLM KD, α=0.5, T=2.0)")
print("="*80)
print("Configuration: hybrid_70m + stage0_biomedlm.yaml")
print("Teacher: BioMedLM (frozen, 2.7B)")
print("Dataset: PubMed (full, streaming)")
print("="*80)

result = subprocess.run(stage0_cmd, shell=True)
if result.returncode == 0:
    print("✓ STAGE 0 TRAINING COMPLETED")
else:
    raise RuntimeError("Stage 0 failed.")

In [ ]:
import os
import torch

ckpt_path = './outputs/colab_stage0_kd_biomedlm_test/checkpoints/last.ckpt'
if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / (1024**2)
    print(f"✓ Stage 0 checkpoint: {size_mb:.1f} MB")
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print(f"  Keys: {len(ckpt.get('state_dict', ckpt))}")
else:
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

## Phase 4: Stage 1 SimCSE + PubMedBERT KD (2k steps, ~40-60 min)

In [ ]:
import subprocess

lm_ckpt = './outputs/colab_stage0_kd_biomedlm_test/checkpoints/last.ckpt'

stage1_cmd = f'''
LM_CHECKPOINT={lm_ckpt} \\
python scripts/train_contrastive.py \\
    --config-name config_70m \\
    dataset=pubmed \\
    trainer=colab_single_gpu \\
    distill=stage1_pubmedbert \\
    contrastive_mode=simcse \\
    trainer.max_steps=2000 \\
    trainer.accumulate_grad_batches=4 \\
    dataset.batch_size=8 \\
    trainer.precision=16-mixed \\
    lm_checkpoint="{lm_ckpt}" \\
    experiment_name=colab_stage1_kd_pubmedbert_test \\
    output_dir=./outputs/colab_stage1_kd_pubmedbert_test \\
    wandb.enabled=false
'''

print("="*80)
print("STAGE 1: SimCSE + PubMedBERT KD (λ ramps 0→0.3)")
print("="*80)
print("Configuration: hybrid_70m + stage1_pubmedbert.yaml")
print("Teacher: PubMedBERT (frozen, 110M)")
print("LM Checkpoint: Stage 0 result")
print("="*80)

result = subprocess.run(stage1_cmd, shell=True)
if result.returncode == 0:
    print("✓ STAGE 1 TRAINING COMPLETED")
else:
    raise RuntimeError("Stage 1 failed.")

In [ ]:
import os
import torch

ckpt_path = './outputs/colab_stage1_kd_pubmedbert_test/checkpoints/last.ckpt'
if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / (1024**2)
    print(f"✓ Stage 1 checkpoint: {size_mb:.1f} MB")
    ckpt = torch.load(ckpt_path, map_location='cpu')
    state = ckpt.get('state_dict', ckpt)
    has_proj = any('projection_head' in k for k in state.keys())
    print(f"  Has projection_head: {has_proj}")
else:
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

## Phase 5: Validation & Inference

In [ ]:
import torch
from hybrid_xmamba.models.configuration_hybrid import HybridConfig
from hybrid_xmamba.models.hybrid_lm import HybridTextEncoder

ckpt = torch.load('./outputs/colab_stage1_kd_pubmedbert_test/checkpoints/last.ckpt', map_location='cpu')
state = ckpt.get('state_dict', ckpt)
state = {k.replace('model.', '', 1): v for k, v in state.items()}

config = HybridConfig(vocab_size=50257, dim=512, num_layers=8, layer_pattern=['mamba', 'mamba', 'mlstm'],
                      max_position_embeddings=1024, num_heads=8, head_dim=64,
                      slstm_hidden_dim=512, slstm_num_heads=4)

encoder = HybridTextEncoder(config, embed_dim=512).eval()
encoder.load_state_dict(state, strict=False)
encoder.to('cuda')

print("✓ Checkpoint loaded")
print(f"  Model: {encoder.__class__.__name__}")
print(f"  Parameters: {sum(p.numel() for p in encoder.parameters())/1e6:.1f}M")

In [ ]:
from transformers import AutoTokenizer
import torch

texts = [
    "Machine learning in genomic prediction",
    "Deep learning for natural language processing",
    "Transformers in biomedical applications",
]

tokenizer = AutoTokenizer.from_pretrained('gpt2')
with torch.no_grad():
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
    inputs = {k: v.to('cuda') for k, v in inputs.items()}
    embeddings = encoder.encode(inputs['input_ids'], attention_mask=inputs.get('attention_mask')).cpu()

print(f"✓ Generated embeddings: {embeddings.shape}")
print(f"  Dtype: {embeddings.dtype}")

In [ ]:
import torch

print("Embedding Quality Checks:")
print("="*60)

norms = torch.norm(embeddings, dim=1)
print(f"L2-Norm: {norms.tolist()}")
print(f"  All ~1.0? {all(0.99 <= n <= 1.01 for n in norms)}")

has_nan = torch.isnan(embeddings).any()
has_inf = torch.isinf(embeddings).any()
print(f"NaN/Inf: nan={has_nan}, inf={has_inf}")

cos_sims = embeddings @ embeddings.T
print(f"Cosine Sims:\n{cos_sims.numpy()}")

off_diag = cos_sims[~torch.eye(3, dtype=torch.bool)]
max_sim = off_diag.max()
print(f"Max off-diag: {max_sim:.4f}")
print(f"No collapse? {max_sim < 0.95}")
print("="*60)

## Final Summary

In [ ]:
import os

print("\n" + "="*80)
print("VALIDATION COMPLETE ✓")
print("="*80)
print("\nCheckpoints:")
s0 = './outputs/colab_stage0_kd_biomedlm_test/checkpoints/last.ckpt'
s1 = './outputs/colab_stage1_kd_pubmedbert_test/checkpoints/last.ckpt'
if os.path.exists(s0): print(f"  Stage 0: {os.path.getsize(s0)/(1024**2):.1f} MB")
if os.path.exists(s1): print(f"  Stage 1: {os.path.getsize(s1)/(1024**2):.1f} MB")
print("\nNext: Transfer Stage 1 checkpoint to A100 for production training")
print("  - Increase steps: 5k→40k (Stage 0), 2k→10k (Stage 1)")
print("  - Increase batch: 8→32")
print("  - Use bf16-mixed precision + torch.compile")
print("="*80)